# 🔬 Sensibilité au volume de CSV — Modèle Mixte
## Consigne Dr. Sarun (Training Procedure -> Best Score, point 1)
### CMKL University · Stage 2026

---

**Question posée** : combien de spectres CSV sont réellement nécessaires ?
Comment la réduction du volume d'entraînement CSV impacte-t-elle les résultats ?

**Protocole** :
```
Test CSV  : TOUJOURS IDENTIQUE (même 15% réservé, même seed, jamais retouché)
Val CSV   : TOUJOURS IDENTIQUE (même 15% réservé)
Train CSV : fraction VARIABLE du pool disponible (70% → 50% → 30% → 10% ...)
```

**Point méthodologique important** : les sous-ensembles sont **emboîtés**
(nested) — le train à 30% est un sous-ensemble du train à 50%, qui est lui-même
un sous-ensemble du train à 70%. Ça isole proprement l'effet du VOLUME de
données, sans confondre avec un changement de composition.

**Un seul paramètre à changer entre chaque run** :
```python
CFG['csv_train_fraction'] = 1.0   # 100% du pool train disponible (≈70% du total CSV)
CFG['csv_train_fraction'] = 0.7   # 70% du pool  → ≈50% du total CSV
CFG['csv_train_fraction'] = 0.43  # 43% du pool  → ≈30% du total CSV
CFG['csv_train_fraction'] = 0.14  # 14% du pool  → ≈10% du total CSV
```
Le reste du notebook n'a rien à changer — relance simplement Run All après
avoir modifié cette seule ligne.


---
## ⚙️ Section 0 — Imports & Configuration


In [103]:
import subprocess, sys
def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
for pkg in ['scikit-learn', 'seaborn']:
    try: __import__(pkg.replace('-','_'))
    except ImportError: install(pkg)
print('✓ Packages prêts')

✓ Packages prêts


In [104]:
import os, glob, re, io, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, OneCycleLR

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
HOME   = os.path.expanduser('~')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

Device : cuda
GPU    : NVIDIA A100-SXM4-40GB


In [105]:
# ════════════════════════════════════════════════════════════════════
# 🎛️  PARAMÈTRE À CHANGER ENTRE CHAQUE RUN — tout le reste est fixe
# ════════════════════════════════════════════════════════════════════
CSV_TRAIN_FRACTION = 0.07   # ← MODIFIE UNIQUEMENT CETTE LIGNE
# Valeurs suggérées pour le sweep complet :
#   1.0   → 100% du pool train (≈70% du total CSV) — run de référence
#   0.7   → 70% du pool        (≈50% du total CSV)
#   0.43  → 43% du pool        (≈30% du total CSV)
#   0.14  → 14% du pool        (≈10% du total CSV)
#   0.07  → 7% du pool         (≈5% du total CSV)
# ════════════════════════════════════════════════════════════════════

RUN_LABEL = f'csvfrac{int(CSV_TRAIN_FRACTION*100)}'
print(f'🎛️  Run actuel : {RUN_LABEL}  (CSV_TRAIN_FRACTION = {CSV_TRAIN_FRACTION})')

🎛️  Run actuel : csvfrac7  (CSV_TRAIN_FRACTION = 0.07)


In [106]:
from torch.utils.tensorboard import SummaryWriter

LOG_DIR = os.path.join(HOME, 'runs', f'patchtst_csv_sensitivity_{RUN_LABEL}')
os.makedirs(LOG_DIR, exist_ok=True)
writer  = SummaryWriter(LOG_DIR)
print(f'✓ TensorBoard logs → {LOG_DIR}')

def safe_log(writer, *args, method='add_scalar', **kwargs):
    try: getattr(writer, method)(*args, **kwargs)
    except Exception: pass

✓ TensorBoard logs → /home/glider/runs/patchtst_csv_sensitivity_csvfrac7


In [107]:
# ════════════════════════════════════════════════════════════════════
# CONFIG — architecture et hyperparamètres FIXES (config optimale confirmée)
# ════════════════════════════════════════════════════════════════════
CFG = {
    'L'         : 6700,
    'WN_MIN'    : 650,
    'WN_MAX'    : 4000,
    'WN_STEP'   : 0.5,

    'patch_size' : 320,
    'stride'     : 256,

    'd_model'   : 256,
    'n_heads'   : 16,
    'n_layers'  : 5,
    'd_ff'      : 512,
    'dropout'   : 0.1,

    'mask_ratio'  : 0.40,
    'ssl_epochs'  : 100,
    'ssl_lr'      : 1e-3,
    'ssl_batch'   : 64,

    'alpha'        : 1.0,
    'beta'         : 0.5,
    'probe_epochs' : 80,
    'probe_lr'     : 1e-3,
    'ft_epochs'    : 80,
    'ft_lr'        : 1e-5,
    'batch_size'   : 32,

    'csv_train_fraction' : CSV_TRAIN_FRACTION,

    'ssl_path'   : os.path.join(HOME, 'models', f'ssl_backbone_{RUN_LABEL}.pth'),
    'probe_path' : os.path.join(HOME, 'models', f'probe_model_{RUN_LABEL}.pth'),
    'final_path' : os.path.join(HOME, 'models', f'final_model_{RUN_LABEL}.pth'),

    'noise_variant' : 'Upto30SNR',
}
os.makedirs(os.path.join(HOME, 'models'), exist_ok=True)

WN_GRID   = np.arange(CFG['WN_MIN'], CFG['WN_MAX'], CFG['WN_STEP'])
CFG['L']  = len(WN_GRID)
L = CFG['L']
N_PATCHES = (L - CFG['patch_size']) // CFG['stride'] + 2
print(f'L = {L}, N_PATCHES = {N_PATCHES}')
assert CFG['d_model'] % CFG['n_heads'] == 0

L = 6700, N_PATCHES = 26


In [108]:
ASSUMED_CLASSES = [
    'ABS', 'ACRYLIC', 'CELLULOSE', 'CHITOSAN', 'ENR', 'EPDM', 'EVA', 'HDPE',
    'LDPE', 'NYLON', 'PBAT', 'PBS', 'PC', 'PEEK', 'PEI', 'PET',
    'PF THERMOPLASTIC', 'PF THERMOSET', 'PHB', 'PLA', 'PMMA', 'POM', 'PP',
    'PS', 'PTFE', 'PU', 'PVA', 'PVC', 'PVDF', 'SAN',
]
N_CLASSES = len(ASSUMED_CLASSES)
CFG['N_CLASSES'] = N_CLASSES
le = LabelEncoder()
le.fit(ASSUMED_CLASSES)
print(f'{N_CLASSES} classes')

30 classes


---
## 📊 Section 1 — Données .npy (TOUJOURS COMPLÈTES, non affectées par le sweep)


In [109]:
NPY_ROOT  = os.path.join(HOME, 'data', '2026-FTIR-Preprocesed','2026 - FTIR - 4. Selected Datasets - Preprocessed')
TRAIN_DIR = os.path.join(NPY_ROOT, '1.1 TrainingSet - UptoY dB')
TEST_DIR  = os.path.join(NPY_ROOT, '1.2 TestSet - UptoY dB')

def npy_path(base_dir, filename):
    p = os.path.join(base_dir, filename)
    if not os.path.exists(p): print(f'  ✗ INTROUVABLE : {p}')
    return p

noise = CFG['noise_variant']
npy_train_clean = np.load(npy_path(TRAIN_DIR, 'TrainGroundTruthSet_Pre.npy'))
npy_train_noisy = np.load(npy_path(TRAIN_DIR, f'TrainNoisySet_{noise}_Pre.npy'))
npy_test_clean  = np.load(npy_path(TEST_DIR,  'TestGroundTruthSet_Pre.npy'))
npy_test_noisy  = np.load(npy_path(TEST_DIR,  f'TestNoisySet_{noise}_Pre.npy'))

assert npy_train_clean.shape[1] == L
N_PER_CLASS_NPY_TRAIN = npy_train_clean.shape[0] // N_CLASSES
N_PER_CLASS_NPY_TEST  = npy_test_clean.shape[0]  // N_CLASSES
npy_labels_train = np.repeat(np.arange(N_CLASSES), N_PER_CLASS_NPY_TRAIN)
npy_labels_test  = np.repeat(np.arange(N_CLASSES), N_PER_CLASS_NPY_TEST)

print(f'✓ .npy Train : {npy_train_noisy.shape[0]} spectres  (INCHANGÉ quel que soit le sweep CSV)')
print(f'✓ .npy Test  : {npy_test_noisy.shape[0]} spectres')

✓ .npy Train : 18000 spectres  (INCHANGÉ quel que soit le sweep CSV)
✓ .npy Test  : 18000 spectres


---
## 📁 Section 2 — Données CSV avec split emboîté (nested)

**Étape A** : split fixe Val/Test (identique à chaque run, seed fixe).  
**Étape B** : à partir du pool Train restant, on prend un **sous-ensemble
emboîté** de taille `CSV_TRAIN_FRACTION × pool` — mélangé une seule fois avec
une seed fixe, puis on prend un préfixe de longueur variable.


In [110]:
CSV_ROOT = os.path.join(HOME, 'data', '2026-FirstDataSet', '2026 - Complete FTIR Dataset')

PATHS_CSV = {
    '2023_base' : os.path.join(CSV_ROOT, '2023 Dataset - 22 MP Types with 10 Clean and 60 Noisy'),
    '2025_ext'  : os.path.join(CSV_ROOT, '2025 Dataset 1 - Same 22 MP Types - Add 40 Spectra'),
    '2025_new'  : os.path.join(CSV_ROOT, '2025 Dataset 2 - New 9 MP Types - 50 Clean and 100 Noisy'),
}
for name, path in PATHS_CSV.items():
    status = '✓' if os.path.exists(path) else '✗ INTROUVABLE'
    print(f'  {status}  {name}')

  ✓  2023_base
  ✓  2025_ext
  ✓  2025_new


In [111]:
EXCLUDE_FILES = {'ref.csv', 'reference.csv', 'background.csv', 'bg.csv'}

def is_noisy_csv(filepath):
    p = str(filepath).lower()
    if any(k in p for k in ['noisy', '_sd', '-sd', 'sd_']): return True
    if any(k in p for k in ['clean', '_rm', '-rm', 'rm_']): return False
    return False

def extract_label_csv(filepath):
    name = Path(filepath).stem.upper()
    for pattern in ['_SD_', '_RM_', '_NOISY', '_CLEAN', 'PARTICLE', '-NOISY',
                    '-CLEAN', '_50', '_60', '_40', '_100', '_10', '_30',
                    ' SPECTRUMS', ' SPECTUMS', 'ADD_40', '-ADD_40']:
        name = name.replace(pattern, ' ')
    name = re.sub(r'\d+', '', name)
    name = re.sub(r'\bNEW\b|\bJAN\b|\bX\b', '', name)
    name = ' '.join(name.replace('_', ' ').replace('-', ' ').split())
    MAPPING = {
        'NYLON PARTICLE' : 'NYLON', 'PTEE' : 'PTFE', 'PTFE' : 'PTFE',
        'PF THERMOPLASTIC CLEAN' : 'PF THERMOPLASTIC',
        'PF THERMOSET CLEAN'     : 'PF THERMOSET',
    }
    if name in MAPPING: return MAPPING[name]
    if name in ASSUMED_CLASSES: return name
    for cls in ASSUMED_CLASSES:
        if cls in name or name in cls: return cls
    return None

def read_csv_multispectra(filepath, sep=','):
    try:
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            lines = f.readlines()
        start_idx = 0
        for i, line in enumerate(lines):
            parts = line.strip().split(sep)
            if len(parts) >= 2:
                try:
                    float(parts[0].replace(',', '.'))
                    start_idx = i; break
                except ValueError: continue
        valid = ''.join(lines[start_idx:])
        headers = lines[start_idx-1].strip().split(sep) if start_idx > 0 else []
        try:
            df = pd.read_csv(io.StringIO(valid), sep=sep, header=None, decimal=',')
        except Exception:
            df = pd.read_csv(io.StringIO(valid), sep=sep, header=None, decimal='.')
        cols = []
        for ci, cn in enumerate(df.columns):
            h = headers[ci].upper() if ci < len(headers) else ''
            v = str(df[cn].iloc[0]).upper()
            if any(k in h for k in ['AIR','BACKGROUND','BG']): continue
            if any(k in v for k in ['AIR','BACKGROUND','BG']): continue
            cols.append(cn)
        df = df[cols].apply(pd.to_numeric, errors='coerce')
        df = df.dropna(subset=[df.columns[0]])
        if len(df) < 100: return None
        wn    = df.iloc[:, 0].values.astype(float)
        order = np.argsort(wn); wn = wn[order]
        spectra = []
        for c in range(1, df.shape[1]):
            ab = df.iloc[order, c].values.astype(float)
            if np.isnan(ab).all() or ab.std() < 1e-10: continue
            nans = np.isnan(ab)
            if nans.any():
                ab[nans] = np.interp(np.where(nans)[0], np.where(~nans)[0], ab[~nans])
            spectra.append(ab.astype(np.float32))
        return (wn, spectra) if spectra else None
    except Exception: return None

print('✓ Fonctions de lecture définies')

✓ Fonctions de lecture définies


In [112]:
print('Chargement des CSV (propres + bruités)...')
csv_records = []
for src_name, folder in PATHS_CSV.items():
    if not os.path.exists(folder): continue
    files = glob.glob(os.path.join(folder, '**/*.csv'), recursive=True)
    n_ok = 0
    for fp in files:
        if Path(fp).name.lower() in EXCLUDE_FILES: continue
        label = extract_label_csv(fp)
        if label is None: continue
        result = read_csv_multispectra(fp)
        if result is None: continue
        wn, spectra_list = result
        noisy = is_noisy_csv(fp)
        for sp in spectra_list:
            sp_interp = np.interp(WN_GRID, wn, sp).astype(np.float32)
            csv_records.append({'label': label, 'is_noisy': noisy, 'spectrum': sp_interp})
            n_ok += 1
    print(f'  ✓ {src_name:12s} : {n_ok:4d} spectres')

df_csv_all = pd.DataFrame(csv_records)
df_csv_all['label_enc'] = le.transform(df_csv_all['label'])
print(f'\n  Total CSV : {len(df_csv_all)} spectres')

Chargement des CSV (propres + bruités)...
  ✓ 2023_base    : 1518 spectres
  ✓ 2025_ext     :  881 spectres
  ✓ 2025_new     : 1145 spectres

  Total CSV : 3544 spectres


In [113]:
df_csv_clean = df_csv_all[~df_csv_all['is_noisy']].reset_index(drop=True)
df_csv_noisy = df_csv_all[ df_csv_all['is_noisy']].reset_index(drop=True)

csv_clean_reference = {}
for cls_idx in range(N_CLASSES):
    subset = df_csv_clean[df_csv_clean['label_enc'] == cls_idx]['spectrum']
    if len(subset) > 0:
        csv_clean_reference[cls_idx] = np.mean(np.stack(subset.values), axis=0).astype(np.float32)
csv_global_clean_mean = (np.mean(np.stack(df_csv_clean['spectrum'].values), axis=0).astype(np.float32)
                         if len(df_csv_clean) > 0 else np.zeros(L, dtype=np.float32))
print(f'Référence propre disponible pour {len(csv_clean_reference)}/{N_CLASSES} classes')

Référence propre disponible pour 21/30 classes


In [114]:
# ── ÉTAPE A : Split fixe Val/Test (SEED FIXE — identique à chaque run) ────
csv_noisy_counts = Counter(df_csv_noisy['label_enc'])
csv_singleton = {k for k, v in csv_noisy_counts.items() if v < 3}
df_csv_multi  = df_csv_noisy[~df_csv_noisy['label_enc'].isin(csv_singleton)]
df_csv_single = df_csv_noisy[ df_csv_noisy['label_enc'].isin(csv_singleton)]

idx_tr_csv, idx_valtest_csv = train_test_split(
    range(len(df_csv_multi)), test_size=0.3,
    random_state=SEED, stratify=df_csv_multi['label_enc'])
idx_val_csv, idx_test_csv = train_test_split(idx_valtest_csv, test_size=0.5, random_state=SEED)

df_csv_train_pool = pd.concat([df_csv_multi.iloc[idx_tr_csv], df_csv_single]).reset_index(drop=True)
df_csv_val   = df_csv_multi.iloc[idx_val_csv].reset_index(drop=True)
df_csv_test  = df_csv_multi.iloc[idx_test_csv].reset_index(drop=True)

print(f'Pool Train complet : {len(df_csv_train_pool)}   (100% = CSV_TRAIN_FRACTION=1.0)')
print(f'Val (FIXE, ne change jamais)  : {len(df_csv_val)}')
print(f'Test (FIXE, ne change jamais) : {len(df_csv_test)}')

Pool Train complet : 1864   (100% = CSV_TRAIN_FRACTION=1.0)
Val (FIXE, ne change jamais)  : 399
Test (FIXE, ne change jamais) : 400


In [115]:
# ── ÉTAPE B : sous-ensemble EMBOÎTÉ du pool train ──────────────────────────
# Mélange une seule fois avec une seed fixe → un préfixe plus court est TOUJOURS
# un sous-ensemble exact d'un préfixe plus long (comparaison propre entre runs).
rng_nested = np.random.RandomState(SEED)
shuffled_idx = rng_nested.permutation(len(df_csv_train_pool))

n_keep = max(1, int(len(df_csv_train_pool) * CFG['csv_train_fraction']))
kept_idx = shuffled_idx[:n_keep]

df_csv_train = df_csv_train_pool.iloc[kept_idx].reset_index(drop=True)

pct_of_total_csv = len(df_csv_train) / len(df_csv_noisy)

print(f'═'*60)
print(f'  CSV_TRAIN_FRACTION = {CFG["csv_train_fraction"]}')
print(f'  → Train CSV effectif : {len(df_csv_train)} spectres')
print(f'    (= {pct_of_total_csv:.1%} du total CSV bruité disponible)')
print(f'═'*60)

# ── Diagnostic : combien de classes restent représentées ? ─────────────────
classes_present = df_csv_train['label_enc'].nunique()
print(f'\n  Classes encore représentées dans le train : {classes_present}/{N_CLASSES}')
if classes_present < N_CLASSES:
    missing = set(range(N_CLASSES)) - set(df_csv_train['label_enc'].unique())
    missing_names = [le.classes_[i] for i in missing]
    print(f'  ⚠️ Classes ABSENTES du train à cette fraction : {missing_names}')
    print(f'     → Ces classes reposeront uniquement sur le signal .npy')

════════════════════════════════════════════════════════════
  CSV_TRAIN_FRACTION = 0.07
  → Train CSV effectif : 130 spectres
    (= 4.9% du total CSV bruité disponible)
════════════════════════════════════════════════════════════

  Classes encore représentées dans le train : 29/30
  ⚠️ Classes ABSENTES du train à cette fraction : [np.str_('PU')]
     → Ces classes reposeront uniquement sur le signal .npy


---
## 🏗️ Section 3 — Datasets & Architecture (identiques au notebook mixte)


In [116]:
class MixedDataset(Dataset):
    def __init__(self, npy_noisy, npy_clean, npy_labels,
                 csv_df, csv_clean_ref, csv_global_mean):
        self.npy_noisy  = npy_noisy.astype(np.float32)
        self.npy_clean  = npy_clean.astype(np.float32)
        self.npy_labels = npy_labels.astype(np.int64)
        self.n_npy = len(npy_labels)
        self.csv_spectra = (np.stack(csv_df['spectrum'].values).astype(np.float32)
                            if len(csv_df) > 0 else np.zeros((0, len(npy_noisy[0])), np.float32))
        self.csv_labels = (csv_df['label_enc'].values.astype(np.int64)
                           if len(csv_df) > 0 else np.zeros(0, np.int64))
        self.csv_clean_ref   = csv_clean_ref
        self.csv_global_mean = csv_global_mean
        self.n_csv = len(self.csv_labels)
    def __len__(self): return self.n_npy + self.n_csv
    def __getitem__(self, idx):
        if idx < self.n_npy:
            x       = torch.tensor(self.npy_noisy[idx], dtype=torch.float32)
            x_clean = torch.tensor(self.npy_clean[idx], dtype=torch.float32)
            y       = torch.tensor(self.npy_labels[idx], dtype=torch.long)
        else:
            i = idx - self.n_npy
            x = torch.tensor(self.csv_spectra[i], dtype=torch.float32)
            y = torch.tensor(self.csv_labels[i], dtype=torch.long)
            ref = self.csv_clean_ref.get(int(self.csv_labels[i]), self.csv_global_mean)
            x_clean = torch.tensor(ref, dtype=torch.float32)
        mu, sigma = x.mean(), x.std() + 1e-8
        x       = (x - mu) / sigma
        x_clean = (x_clean - mu) / sigma
        return x, x_clean, y

class SimpleMultiTaskDataset(Dataset):
    def __init__(self, noisy, clean, labels):
        self.noisy  = noisy.astype(np.float32)
        self.clean  = clean.astype(np.float32)
        self.labels = labels.astype(np.int64)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        x       = torch.tensor(self.noisy[idx], dtype=torch.float32)
        x_clean = torch.tensor(self.clean[idx], dtype=torch.float32)
        y       = torch.tensor(self.labels[idx], dtype=torch.long)
        mu, sigma = x.mean(), x.std() + 1e-8
        x       = (x - mu) / sigma
        x_clean = (x_clean - mu) / sigma
        return x, x_clean, y

train_dataset = MixedDataset(npy_train_noisy, npy_train_clean, npy_labels_train,
                              df_csv_train, csv_clean_reference, csv_global_clean_mean)

csv_val_spectra = np.stack(df_csv_val['spectrum'].values) if len(df_csv_val) > 0 else np.zeros((0,L))
csv_val_clean   = np.stack([csv_clean_reference.get(int(l), csv_global_clean_mean)
                            for l in df_csv_val['label_enc'].values]) if len(df_csv_val) > 0 else np.zeros((0,L))
val_csv_dataset = SimpleMultiTaskDataset(csv_val_spectra, csv_val_clean, df_csv_val['label_enc'].values)

csv_test_spectra = np.stack(df_csv_test['spectrum'].values) if len(df_csv_test) > 0 else np.zeros((0,L))
csv_test_clean   = np.stack([csv_clean_reference.get(int(l), csv_global_clean_mean)
                             for l in df_csv_test['label_enc'].values]) if len(df_csv_test) > 0 else np.zeros((0,L))
test_csv_dataset = SimpleMultiTaskDataset(csv_test_spectra, csv_test_clean, df_csv_test['label_enc'].values)

test_npy_dataset = SimpleMultiTaskDataset(npy_test_noisy, npy_test_clean, npy_labels_test)

print(f'✓ train_dataset (mixte) : {len(train_dataset)} (.npy={train_dataset.n_npy}, CSV={train_dataset.n_csv})')
print(f'✓ val_csv   : {len(val_csv_dataset)}')
print(f'✓ test_csv  : {len(test_csv_dataset)} (FIXE, identique à tous les runs)')
print(f'✓ test_npy  : {len(test_npy_dataset)}')

✓ train_dataset (mixte) : 18130 (.npy=18000, CSV=130)
✓ val_csv   : 399
✓ test_csv  : 400 (FIXE, identique à tous les runs)
✓ test_npy  : 18000


In [117]:
weight_npy = 1.0 / train_dataset.n_npy
weight_csv = 1.0 / max(train_dataset.n_csv, 1)
sample_weights = np.concatenate([
    np.full(train_dataset.n_npy, weight_npy),
    np.full(train_dataset.n_csv, weight_csv),
])
sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.float32),
    num_samples=len(train_dataset), replacement=True)

train_loader    = DataLoader(train_dataset, batch_size=CFG['batch_size'], sampler=sampler, num_workers=0)
val_csv_loader  = DataLoader(val_csv_dataset, batch_size=CFG['batch_size'], shuffle=False, num_workers=0)
test_npy_loader = DataLoader(test_npy_dataset, batch_size=CFG['batch_size'], shuffle=False, num_workers=0)
test_csv_loader = DataLoader(test_csv_dataset, batch_size=CFG['batch_size'], shuffle=False, num_workers=0)

# ── Val .npy (réservé du train .npy, comme d'habitude) ─────────────────────
N_VAL_PER_CLASS_NPY = 30
val_idx_npy, train_idx_npy2 = [], []
for c in range(N_CLASSES):
    cls_idx = np.where(npy_labels_train == c)[0]
    rng = np.random.RandomState(SEED)
    rng.shuffle(cls_idx)
    val_idx_npy.extend(cls_idx[:N_VAL_PER_CLASS_NPY])
val_idx_npy = np.array(val_idx_npy)
val_npy_dataset = SimpleMultiTaskDataset(npy_train_noisy[val_idx_npy], npy_train_clean[val_idx_npy],
                                          npy_labels_train[val_idx_npy])
val_npy_loader = DataLoader(val_npy_dataset, batch_size=CFG['batch_size'], shuffle=False, num_workers=0)

print('✓ DataLoaders prêts')

✓ DataLoaders prêts


In [118]:
csv_all_spectra = np.stack(df_csv_all['spectrum'].values)
class SSLPoolDataset(Dataset):
    def __init__(self, arrays, max_n=None, seed=SEED):
        rng = np.random.RandomState(seed)
        if max_n is not None:
            n_per_array = max_n // len(arrays)
            sampled = []
            for arr in arrays:
                if len(arr) > n_per_array:
                    idx = rng.choice(len(arr), size=n_per_array, replace=False)
                    sampled.append(arr[idx].astype(np.float32))
                else:
                    sampled.append(arr.astype(np.float32))
            self.spectra = np.concatenate(sampled, axis=0)
        else:
            self.spectra = np.concatenate(arrays, axis=0).astype(np.float32)
    def __len__(self): return len(self.spectra)
    def __getitem__(self, idx):
        x = torch.tensor(self.spectra[idx], dtype=torch.float32)
        x = (x - x.mean()) / (x.std() + 1e-8)
        return x

ssl_dataset = SSLPoolDataset(
    arrays=[npy_train_noisy, npy_train_clean, npy_test_noisy, npy_test_clean, csv_all_spectra],
    max_n=35000)
print(f'✓ SSL pool : {len(ssl_dataset)} spectres')

✓ SSL pool : 31544 spectres


In [119]:
class PatchEmbedding(nn.Module):
    def __init__(self, L, patch_size, stride, d_model):
        super().__init__()
        self.P, self.S, self.D = patch_size, stride, d_model
        self.N = (L - patch_size) // stride + 2
        self.patch_proj = nn.Linear(patch_size, d_model)
        self.pos_embed  = nn.Embedding(self.N, d_model)
        self.dropout    = nn.Dropout(0.1)
    def get_raw_patches(self, x):
        B = x.shape[0]
        pad = x[:, -1:].expand(B, self.S)
        x_pad = torch.cat([x, pad], dim=1)
        return x_pad.unfold(1, self.P, self.S)
    def forward(self, x):
        patches  = self.get_raw_patches(x)
        content  = self.patch_proj(patches)
        pos_vecs = self.pos_embed(torch.arange(self.N, device=x.device))
        return self.dropout(content + pos_vecs)

class ConformerFFN(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.W1, self.V, self.W2 = (nn.Linear(d_model, d_ff), nn.Linear(d_model, d_ff),
                                     nn.Linear(d_ff, d_model))
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        x = self.norm(x)
        return self.drop(self.W2(F.silu(self.W1(x)) * self.V(x)))

class ConformerConvModule(nn.Module):
    def __init__(self, d_model, kernel_size=31, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.pw1  = nn.Conv1d(d_model, 2*d_model, 1)
        self.glu  = nn.GLU(dim=1)
        self.dw   = nn.Conv1d(d_model, d_model, kernel_size, padding=kernel_size//2, groups=d_model)
        self.bn   = nn.BatchNorm1d(d_model)
        self.act  = nn.SiLU()
        self.pw2  = nn.Conv1d(d_model, d_model, 1)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        r = x
        x = self.norm(x).transpose(1,2)
        x = self.glu(self.pw1(x))
        x = self.act(self.bn(self.dw(x)))
        x = self.drop(self.pw2(x)).transpose(1,2)
        return r + x

class ConformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout, kernel_size=31):
        super().__init__()
        self.ffn1 = ConformerFFN(d_model, d_ff, dropout)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.attn_norm = nn.LayerNorm(d_model)
        self.conv = ConformerConvModule(d_model, kernel_size, dropout)
        self.ffn2 = ConformerFFN(d_model, d_ff, dropout)
        self.norm = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        x = x + 0.5 * self.ffn1(x)
        xn = self.attn_norm(x)
        x  = x + self.drop(self.attn(xn, xn, xn)[0])
        x  = self.conv(x)
        x  = x + 0.5 * self.ffn2(x)
        return self.norm(x)

class TransformerBackbone(nn.Module):
    def __init__(self, d_model, n_heads, n_layers, d_ff, dropout):
        super().__init__()
        self.layers = nn.ModuleList([
            ConformerBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
    def forward(self, x):
        for l in self.layers: x = l(x)
        return self.norm(x)

class ReconstructionHead(nn.Module):
    def __init__(self, d_model, patch_size):
        super().__init__()
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, d_model),
                                   nn.GELU(), nn.Linear(d_model, patch_size))
    def forward(self, z): return self.head(z)

class ClassificationHead(nn.Module):
    def __init__(self, d_model, n_classes, dropout=0.1, hidden_dim=None):
        super().__init__()
        if hidden_dim is None: hidden_dim = d_model // 2
        self.attn_pool = nn.Linear(d_model, 1)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, n_classes))
    def forward(self, z):
        w = F.softmax(self.attn_pool(z), dim=1)
        return self.head((w * z).sum(dim=1))

class DenoisingHead(nn.Module):
    def __init__(self, d_model, patch_size, n_patches, stride, spectrum_length):
        super().__init__()
        self.P, self.S, self.N, self.L = patch_size, stride, n_patches, spectrum_length
        self.proj = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, d_model),
                                   nn.GELU(), nn.Linear(d_model, patch_size))
    def forward(self, z):
        B = z.shape[0]
        pr = self.proj(z)
        out = torch.zeros(B, self.L+self.S, device=z.device)
        cnt = torch.zeros(self.L+self.S, device=z.device)
        for k in range(self.N):
            s = k * self.S
            out[:, s:s+self.P] += pr[:, k, :]
            cnt[s:s+self.P]    += 1
        return (out / cnt.clamp(min=1))[:, :self.L]

print('✓ Architecture définie')

✓ Architecture définie


In [120]:
class PatchTSTSSL(nn.Module):
    def __init__(self, patch_embed, backbone, recon_head, mask_ratio=0.4):
        super().__init__()
        self.patch_embed = patch_embed
        self.backbone    = backbone
        self.recon_head  = recon_head
        self.mask_ratio  = mask_ratio
        self.mask_token  = nn.Parameter(torch.zeros(1, 1, patch_embed.D))
        nn.init.trunc_normal_(self.mask_token, std=0.02)
    def forward(self, x):
        B = x.shape[0]
        N = self.patch_embed.N
        patches_orig = self.patch_embed.get_raw_patches(x)
        tokens = self.patch_embed(x)
        n_masked = int(N * self.mask_ratio)
        mask = torch.zeros(B, N, dtype=torch.bool, device=x.device)
        for b in range(B):
            max_start = max(1, N - n_masked)
            start = torch.randint(0, max_start, (1,), device=x.device).item()
            mask[b, start:start+n_masked] = True
        pos_vecs = self.patch_embed.pos_embed(torch.arange(N, device=x.device))
        mask_tok_pos = (self.mask_token.to(x.device) + pos_vecs.unsqueeze(0)).expand(B, -1, -1)
        tokens = torch.where(mask.unsqueeze(-1), mask_tok_pos, tokens)
        z = self.backbone(tokens)
        z_masked      = z[mask]
        patches_recon = self.recon_head(z_masked)
        patches_target= patches_orig[mask]
        loss = F.mse_loss(patches_recon, patches_target)
        return loss, mask

class PatchTSTMultiTask(nn.Module):
    def __init__(self, patch_embed, backbone, class_head, denoise_head, alpha=1.0, beta=0.5):
        super().__init__()
        self.patch_embed, self.backbone = patch_embed, backbone
        self.class_head, self.denoise_head = class_head, denoise_head
        self.alpha, self.beta = alpha, beta
    def encode(self, x):
        return self.backbone(self.patch_embed(x))
    def classify(self, x):
        return self.class_head(self.encode(x))
    def forward(self, x, y=None, clean_target=None):
        z = self.encode(x)
        logits   = self.class_head(z)
        denoised = self.denoise_head(z)
        loss = None
        if y is not None and clean_target is not None:
            loss_clf = F.cross_entropy(logits, y, label_smoothing=0.1)
            loss_den = F.mse_loss(denoised, clean_target)
            loss = self.alpha * loss_clf + self.beta * loss_den
        return logits, denoised, loss

patch_embed  = PatchEmbedding(CFG['L'], CFG['patch_size'], CFG['stride'], CFG['d_model']).to(DEVICE)
backbone     = TransformerBackbone(CFG['d_model'], CFG['n_heads'], CFG['n_layers'],
                                    CFG['d_ff'], CFG['dropout']).to(DEVICE)
recon_head   = ReconstructionHead(CFG['d_model'], CFG['patch_size']).to(DEVICE)
class_head   = ClassificationHead(CFG['d_model'], CFG['N_CLASSES'], dropout=0.1).to(DEVICE)
denoise_head = DenoisingHead(CFG['d_model'], CFG['patch_size'], N_PATCHES,
                              CFG['stride'], CFG['L']).to(DEVICE)

total = sum(p.numel() for m in [patch_embed, backbone, class_head, denoise_head]
            for p in m.parameters() if p.requires_grad)
print(f'Total paramètres : {total:,}')

Total paramètres : 6,596,191


---
## 🛠️ Section 4 — Fonctions d'entraînement


In [121]:
def ssl_train_epoch(model, loader, optimizer, scheduler=None):
    model.train()
    total_loss, n = 0.0, 0
    for x in loader:
        x = x.to(DEVICE)
        loss, _ = model(x)
        optimizer.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        if scheduler: scheduler.step()
        total_loss += loss.item() * len(x); n += len(x)
    return total_loss / n

@torch.no_grad()
def ssl_eval_epoch(model, loader):
    model.eval()
    total_loss, n = 0.0, 0
    for x in loader:
        x = x.to(DEVICE)
        loss, _ = model(x)
        total_loss += loss.item() * len(x); n += len(x)
    return total_loss / n

def snr_db(clean, signal):
    noise_power  = ((signal - clean) ** 2).mean(dim=-1) + 1e-8
    signal_power = (clean ** 2).mean(dim=-1) + 1e-8
    return 10 * torch.log10(signal_power / noise_power)

def multitask_train_epoch(model, loader, optimizer):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    total_mse, total_snr = 0.0, 0.0
    for x, x_clean, y in loader:
        x, x_clean, y = x.to(DEVICE), x_clean.to(DEVICE), y.to(DEVICE)
        logits, denoised, loss = model(x, y=y, clean_target=x_clean)
        optimizer.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        with torch.no_grad():
            mse = F.mse_loss(denoised, x_clean)
            snr_gain = (snr_db(x_clean, denoised) - snr_db(x_clean, x)).mean()
        total_loss += loss.item()*len(y); correct += (logits.argmax(1)==y).sum().item()
        total_mse += mse.item()*len(y); total_snr += snr_gain.item()*len(y); n += len(y)
    return total_loss/n, correct/n, total_mse/n, total_snr/n

@torch.no_grad()
def multitask_eval_epoch(model, loader):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    total_mse, total_snr = 0.0, 0.0
    for x, x_clean, y in loader:
        x, x_clean, y = x.to(DEVICE), x_clean.to(DEVICE), y.to(DEVICE)
        logits, denoised, loss = model(x, y=y, clean_target=x_clean)
        mse = F.mse_loss(denoised, x_clean)
        snr_gain = (snr_db(x_clean, denoised) - snr_db(x_clean, x)).mean()
        total_loss += loss.item()*len(y); correct += (logits.argmax(1)==y).sum().item()
        total_mse += mse.item()*len(y); total_snr += snr_gain.item()*len(y); n += len(y)
    return total_loss/n, correct/n, total_mse/n, total_snr/n

print('✓ Fonctions définies')

✓ Fonctions définies


---
## 🧠 Phase 1 — SSL Pre-training


In [122]:
ssl_model = PatchTSTSSL(patch_embed, backbone, recon_head, CFG['mask_ratio']).to(DEVICE)
patch_embed.dropout.p = 0.0

n_ssl = len(ssl_dataset)
n_val_ssl = int(0.2 * n_ssl)
ssl_train_sub, ssl_val_sub = torch.utils.data.random_split(
    ssl_dataset, [n_ssl-n_val_ssl, n_val_ssl],
    generator=torch.Generator().manual_seed(SEED))
ssl_train_loader = DataLoader(ssl_train_sub, batch_size=CFG['ssl_batch'], shuffle=True, num_workers=0)
ssl_val_loader   = DataLoader(ssl_val_sub, batch_size=CFG['ssl_batch'], shuffle=False, num_workers=0)

ssl_optimizer = AdamW(ssl_model.parameters(), lr=CFG['ssl_lr'], weight_decay=1e-4)
ssl_scheduler = OneCycleLR(ssl_optimizer, max_lr=CFG['ssl_lr'], epochs=CFG['ssl_epochs'],
                           steps_per_epoch=len(ssl_train_loader),
                           pct_start=0.1, div_factor=25.0, final_div_factor=1e4)

best_ssl_loss = float('inf')
print(f'=== Phase 1 : SSL Pre-training — run "{RUN_LABEL}" ({CFG["ssl_epochs"]} époques) ===')
for epoch in range(1, CFG['ssl_epochs']+1):
    tr = ssl_train_epoch(ssl_model, ssl_train_loader, ssl_optimizer, ssl_scheduler)
    va = ssl_eval_epoch(ssl_model, ssl_val_loader)
    safe_log(writer, 'Phase1_SSL/Loss', {'Train': tr, 'Val': va}, epoch, method='add_scalars')
    if va < best_ssl_loss: best_ssl_loss = va
    if epoch % 20 == 0 or epoch == 1:
        print(f'  epoch {epoch:3d} : train={tr:.6f}  val={va:.6f}')

patch_embed.dropout.p = 0.1
torch.save({'patch_embed': patch_embed.state_dict(), 'backbone': backbone.state_dict(),
            'cfg': CFG, 'ssl_val_loss': best_ssl_loss}, CFG['ssl_path'])
print(f'\n✓ Meilleure val MSE : {best_ssl_loss:.6f}')
print(f'✓ Sauvegardé → {CFG["ssl_path"]}')

=== Phase 1 : SSL Pre-training — run "csvfrac7" (100 époques) ===


  epoch   1 : train=0.458628  val=0.217920
  epoch  20 : train=0.015358  val=0.029005
  epoch  40 : train=0.007908  val=0.007681
  epoch  60 : train=0.004531  val=0.004342
  epoch  80 : train=0.002789  val=0.002757
  epoch 100 : train=0.002369  val=0.002871

✓ Meilleure val MSE : 0.002263
✓ Sauvegardé → /home/glider/models/ssl_backbone_csvfrac7.pth


---
## 🔍 Phase 2 — Linear Probing


In [123]:
checkpoint = torch.load(CFG['ssl_path'], map_location=DEVICE, weights_only=False)
patch_embed.load_state_dict(checkpoint['patch_embed'])
backbone.load_state_dict(checkpoint['backbone'])

class_head   = ClassificationHead(CFG['d_model'], CFG['N_CLASSES'], dropout=0.1).to(DEVICE)
denoise_head = DenoisingHead(CFG['d_model'], CFG['patch_size'], N_PATCHES,
                              CFG['stride'], CFG['L']).to(DEVICE)
for p in patch_embed.parameters(): p.requires_grad = False
for p in backbone.parameters():    p.requires_grad = False

probe_model = PatchTSTMultiTask(patch_embed, backbone, class_head, denoise_head,
                                 alpha=CFG['alpha'], beta=CFG['beta']).to(DEVICE)
probe_optimizer = AdamW(filter(lambda p: p.requires_grad, probe_model.parameters()),
                        lr=CFG['probe_lr'], weight_decay=1e-4)

best_probe_acc = 0.0
print(f'=== Phase 2 : Linear Probing — run "{RUN_LABEL}" ({CFG["probe_epochs"]} époques) ===')
for epoch in range(1, CFG['probe_epochs']+1):
    tr_loss, tr_acc, tr_mse, tr_snr = multitask_train_epoch(probe_model, train_loader, probe_optimizer)
    _, va_npy_acc, _, va_npy_snr = multitask_eval_epoch(probe_model, val_npy_loader)
    _, va_csv_acc, _, va_csv_snr = multitask_eval_epoch(probe_model, val_csv_loader)

    safe_log(writer, 'Phase2_Probe/Acc_npy', {'Train': tr_acc, 'Val': va_npy_acc}, epoch, method='add_scalars')
    safe_log(writer, 'Phase2_Probe/Acc_csv', {'Val': va_csv_acc}, epoch, method='add_scalars')

    avg_acc = (va_npy_acc + va_csv_acc) / 2
    if avg_acc > best_probe_acc:
        best_probe_acc = avg_acc
        torch.save(probe_model.state_dict(), CFG['probe_path'])

    if epoch % 20 == 0 or epoch == 1:
        print(f'  epoch {epoch:3d} : train={tr_acc:.2%}  val_npy={va_npy_acc:.2%}  val_csv={va_csv_acc:.2%}')

print(f'\n✓ Meilleure moyenne (npy+CSV)/2 : {best_probe_acc:.2%}')
print(f'✓ Sauvegardé → {CFG["probe_path"]}')

=== Phase 2 : Linear Probing — run "csvfrac7" (80 époques) ===
  epoch   1 : train=46.39%  val_npy=25.00%  val_csv=74.19%


  epoch  20 : train=78.41%  val_npy=80.67%  val_csv=80.70%
  epoch  40 : train=80.90%  val_npy=88.56%  val_csv=79.95%
  epoch  60 : train=82.53%  val_npy=88.78%  val_csv=80.45%
  epoch  80 : train=82.99%  val_npy=89.89%  val_csv=80.70%

✓ Meilleure moyenne (npy+CSV)/2 : 85.74%
✓ Sauvegardé → /home/glider/models/probe_model_csvfrac7.pth


---
## 🎯 Phase 3 — Fine-tuning complet


In [124]:
probe_model.load_state_dict(torch.load(CFG['probe_path'], map_location=DEVICE, weights_only=False))
for p in probe_model.parameters(): p.requires_grad = True

ft_optimizer = AdamW([
    {'params': probe_model.patch_embed.parameters(),  'lr': CFG['ft_lr']},
    {'params': probe_model.backbone.parameters(),     'lr': CFG['ft_lr']},
    {'params': probe_model.class_head.parameters(),   'lr': CFG['ft_lr']*10},
    {'params': probe_model.denoise_head.parameters(), 'lr': CFG['ft_lr']*10},
], weight_decay=1e-4)
ft_scheduler = CosineAnnealingLR(ft_optimizer, T_max=CFG['ft_epochs'], eta_min=1e-6)

best_ft_acc = 0.0
print(f'=== Phase 3 : Fine-tuning — run "{RUN_LABEL}" ({CFG["ft_epochs"]} époques) ===')
for epoch in range(1, CFG['ft_epochs']+1):
    tr_loss, tr_acc, tr_mse, tr_snr = multitask_train_epoch(probe_model, train_loader, ft_optimizer)
    _, va_npy_acc, _, va_npy_snr = multitask_eval_epoch(probe_model, val_npy_loader)
    _, va_csv_acc, _, va_csv_snr = multitask_eval_epoch(probe_model, val_csv_loader)
    ft_scheduler.step()

    safe_log(writer, 'Phase3_FT/Acc_npy', {'Train': tr_acc, 'Val': va_npy_acc}, epoch, method='add_scalars')
    safe_log(writer, 'Phase3_FT/Acc_csv', {'Val': va_csv_acc}, epoch, method='add_scalars')

    if epoch % 5 == 0:
        torch.save({'model_state': probe_model.state_dict(), 'epoch': epoch, 'cfg': CFG},
                   os.path.join(HOME, 'models', f'checkpoint_{RUN_LABEL}_latest.pth'))

    avg_acc = (va_npy_acc + va_csv_acc) / 2
    if avg_acc > best_ft_acc:
        best_ft_acc = avg_acc
        torch.save({'model_state': probe_model.state_dict(), 'cfg': CFG,
                    'le_classes': le.classes_, 'val_acc': best_ft_acc}, CFG['final_path'])

    if epoch % 10 == 0 or epoch == 1:
        print(f'  epoch {epoch:3d} : train={tr_acc:.2%}  val_npy={va_npy_acc:.2%}  val_csv={va_csv_acc:.2%}')

writer.flush()
print(f'\n✓ Meilleure moyenne (npy+CSV)/2 : {best_ft_acc:.2%}')
print(f'✓ Sauvegardé → {CFG["final_path"]}')
print(f'🔴 TÉLÉCHARGE {os.path.basename(CFG["final_path"])} vers ton Mac')

=== Phase 3 : Fine-tuning — run "csvfrac7" (80 époques) ===
  epoch   1 : train=84.34%  val_npy=91.78%  val_csv=80.45%


  epoch  10 : train=90.35%  val_npy=94.56%  val_csv=80.95%
  epoch  20 : train=93.01%  val_npy=95.22%  val_csv=80.20%
  epoch  30 : train=94.51%  val_npy=96.33%  val_csv=80.45%
  epoch  40 : train=95.02%  val_npy=97.11%  val_csv=81.70%
  epoch  50 : train=95.66%  val_npy=97.22%  val_csv=80.70%
  epoch  60 : train=95.70%  val_npy=97.33%  val_csv=80.70%
  epoch  70 : train=95.95%  val_npy=97.56%  val_csv=80.45%
  epoch  80 : train=95.98%  val_npy=97.67%  val_csv=80.70%

✓ Meilleure moyenne (npy+CSV)/2 : 89.43%
✓ Sauvegardé → /home/glider/models/final_model_csvfrac7.pth
🔴 TÉLÉCHARGE final_model_csvfrac7.pth vers ton Mac


---
## 📊 Évaluation finale — sur les deux Test sets FIXES


In [125]:
ckpt = torch.load(CFG['final_path'], map_location=DEVICE, weights_only=False)
probe_model.load_state_dict(ckpt['model_state'])
probe_model.eval()

def evaluate_full(model, loader):
    all_preds, all_labels = [], []
    all_snr_before, all_snr_after = [], []
    with torch.no_grad():
        for x, x_clean, y in loader:
            x, x_clean = x.to(DEVICE), x_clean.to(DEVICE)
            logits, denoised, _ = model(x)
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(y.numpy())
            all_snr_before.append(snr_db(x_clean, x).cpu().numpy())
            all_snr_after.append(snr_db(x_clean, denoised).cpu().numpy())
    return (np.array(all_preds), np.array(all_labels),
            np.concatenate(all_snr_before), np.concatenate(all_snr_after))

preds_npy, labels_npy, snr_b_npy, snr_a_npy = evaluate_full(probe_model, test_npy_loader)
acc_npy = accuracy_score(labels_npy, preds_npy)

preds_csv, labels_csv, snr_b_csv, snr_a_csv = evaluate_full(probe_model, test_csv_loader)
acc_csv = accuracy_score(labels_csv, preds_csv)

print('═'*65)
print(f'  RÉSULTAT — {RUN_LABEL}  (CSV_TRAIN_FRACTION={CFG["csv_train_fraction"]})')
print(f'  Train CSV effectif : {len(df_csv_train)} spectres ({pct_of_total_csv:.1%} du total)')
print('═'*65)
print(f'  Test .npy (officiel, FIXE) : {acc_npy:.2%}   SNR : {(snr_a_npy-snr_b_npy).mean():+.2f}dB')
print(f'  Test CSV (réel, FIXE)      : {acc_csv:.2%}   SNR : {(snr_a_csv-snr_b_csv).mean():+.2f}dB')
print('═'*65)

═════════════════════════════════════════════════════════════════
  RÉSULTAT — csvfrac7  (CSV_TRAIN_FRACTION=0.07)
  Train CSV effectif : 130 spectres (4.9% du total)
═════════════════════════════════════════════════════════════════
  Test .npy (officiel, FIXE) : 94.85%   SNR : +9.34dB
  Test CSV (réel, FIXE)      : 81.25%   SNR : +9.79dB
═════════════════════════════════════════════════════════════════


In [126]:
# ── Logger ce run dans le dashboard HParams dédié au sweep CSV ────────────
from torch.utils.tensorboard import SummaryWriter as SW2
from datetime import datetime

HPARAMS_CSV_SWEEP = os.path.join(HOME, 'runs', 'hparams_csv_sensitivity')

def log_run(cfg, metrics, run_label):
    run_dir = os.path.join(HPARAMS_CSV_SWEEP, run_label)
    w = SW2(run_dir)
    hparams_clean = {k: v for k, v in cfg.items() if isinstance(v, (int, float, str, bool))}
    w.add_hparams(hparams_clean, metrics)
    w.close()
    print(f'✓ Run "{run_label}" loggé dans le dashboard sweep CSV')

log_run(
    cfg={
        'csv_train_fraction' : CFG['csv_train_fraction'],
        'csv_train_n_spectra': len(df_csv_train),
        'csv_pct_of_total'   : round(pct_of_total_csv, 4),
    },
    metrics={
        'test_npy_acc' : acc_npy,
        'test_csv_acc' : acc_csv,
        'avg_acc'      : (acc_npy + acc_csv) / 2,
    },
    run_label=RUN_LABEL,
)
print(f'\nPour visualiser le sweep complet :')
print(f'  tensorboard --logdir {HPARAMS_CSV_SWEEP} --port 6008')

✓ Run "csvfrac7" loggé dans le dashboard sweep CSV

Pour visualiser le sweep complet :
  tensorboard --logdir /home/glider/runs/hparams_csv_sensitivity --port 6008


In [127]:
print('═'*65)
print(f'  ✓ RUN "{RUN_LABEL}" TERMINÉ')
print('═'*65)
print(f'  Prochaine étape : modifie CSV_TRAIN_FRACTION en haut du notebook')
print(f'  et relance Run All pour le prochain point du sweep.')
print()
print(f'  Valeurs suggérées restantes à tester :')
suggested = [1.0, 0.7, 0.43, 0.14, 0.07]
for v in suggested:
    marker = ' ← déjà fait (ce run)' if abs(v - CFG['csv_train_fraction']) < 1e-6 else ''
    print(f'    CSV_TRAIN_FRACTION = {v}{marker}')

═════════════════════════════════════════════════════════════════
  ✓ RUN "csvfrac7" TERMINÉ
═════════════════════════════════════════════════════════════════
  Prochaine étape : modifie CSV_TRAIN_FRACTION en haut du notebook
  et relance Run All pour le prochain point du sweep.

  Valeurs suggérées restantes à tester :
    CSV_TRAIN_FRACTION = 1.0
    CSV_TRAIN_FRACTION = 0.7
    CSV_TRAIN_FRACTION = 0.43
    CSV_TRAIN_FRACTION = 0.14
    CSV_TRAIN_FRACTION = 0.07 ← déjà fait (ce run)
